**Eye-Tracking & Neurocognitive Intelligence | Eye-Tracking Foundations & Signal Analysis | Presented by: Reza Saadatyar (2026)**

**06 | I-VT Threshold Sensitivity and Vendor-Reference Coverage**

![Recorded threshold-dependent I-VT labels and vendor-reference support](../../images/06_detector_sensitivity_and_vendor_reference.svg)

**Objective:** Evaluate how the **prespecified 20, 30 and 40 dva/s I-VT speed thresholds** affect sample-level classifications and conditional agreement with available EyeLink parser labels, without retraining, revisiting gaze preprocessing or selecting a threshold for favourable results.

**Executed dataset:** GazeBase v2.0, Round 1, Session 1: ten participants, four tasks and **40 source recordings**. Notebook 06 produced **120 record-by-threshold rows** from Notebook 02's saved prepared samples. Notebook 03's original I-VT analysis used the **30-dva/s threshold**, allowing a direct reproducibility check.

**Observed record-averaged class fractions (20 / 30 / 40 dva/s):** fixation **28.53% / 61.23% / 78.65%**; unknown **62.24% / 30.57% / 13.78%**. Mean *conditional* EyeLink agreement was **90.97% / 98.13% / 98.78%**, respectively.

**Critical interpretation:** Agreement uses only samples with recognised vendor labels **and** evaluable I-VT fixation/saccade assignments. Its denominator changes with threshold. The published record means are **unweighted across records**, not participant-disjoint validation or detector accuracy. Rising agreement does **not** establish that the largest threshold is best.

**Additional diagnostic:** A compact denominator-and-coverage review and 30-dva/s baseline reconciliation with Notebook 03 are added below. These **new checks have not yet been executed**; original class assignments and 120 source results are preserved.


**00. Colab Repository Setup**

In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


/content/Eye-Tracking-Neurocognitive-Intelligence


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_tracking_foundations")

# ----------------- 2. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

# ------------------------ 4. Study Research Utilities ------------------------
from core.stage_outputs import ensure_stage_inputs
from gaze_source import read_stage
from eye_events import ivt_labels, label_agreement

# --------------------- 5. Output and Presentation Setup ----------------------
output_dir = root / "eye_tracking_foundations" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(
    font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5,
    label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(10, 4),
)
print("Current outputs:", output_dir)


Repository updated.
Current outputs: /content/Eye-Tracking-Neurocognitive-Intelligence/eye_tracking_foundations/output


**02. Recover Prepared Motion and Reference Coverage**

Reuse `02_preprocessed_samples.csv`, which contains the original record identifiers, within-run velocity, timing, validity and vendor parser label fields, alongside `03_vendor_agreement.csv` from the original **30-dva/s** Notebook 03 execution. This notebook does not read the original GazeBase CSVs, reapply the median filter or optimise any threshold.

The saved input spans **40 recordings from 10 participants**. The prepared CSV is large, so it is loaded only for the already-executed threshold comparison; the additional evidence checks introduced in Section 04A read **small saved CSVs** and do not reload or reclassify the multi-million-row gaze table.

The input-recovery cell intentionally prepares the source tables without displaying thousands of rows; coverage and diagnostic results are shown by subsequent analysis cells.


In [3]:
# ============ 02. Recover Prepared Motion and Reference Coverage =============

ensure_stage_inputs(output_dir,2,("02_preprocessed_samples.csv",))
ensure_stage_inputs(output_dir,3,("03_vendor_agreement.csv",))
prepared=read_stage(output_dir/"02_preprocessed_samples.csv",
                    ("record_id","participant_id","n","velocity_dva_s","segment_id","lab"))
reference=read_stage(output_dir/"03_vendor_agreement.csv",
                     ("record_id","vendor_coverage_fraction"))


**03. Evaluate Three Prespecified I-VT Thresholds**

Run exactly **20, 30 and 40 dva/s**, applying the same minimum run durations as Notebook 03: 80 ms for fixation and 6 ms for saccade. Within each of the **40 source records**, retain three fractions: I-VT fixation, I-VT saccade and `unknown`. Invalid or insufficient-duration samples remain `unknown`, not saccades or false positives.

Compare each classification with available EyeLink parser labels using conditional sample-level agreement. Report `eligible_samples`, vendor coverage, the fraction of all samples meeting the joint eligibility condition, and `sample_agreement` **for each record and threshold**. The saved execution contains **120 rows**. As threshold increases, both the classified-sample distribution and the comparison denominator can change.

No setting has been selected by maximising agreement; these are **exploratory sensitivity settings**, not three independent tests of clinical, reading or neurocognitive performance.


In [4]:
# ================ 03. Evaluate Predefined Velocity Thresholds ================

THRESHOLDS=(20.0,30.0,40.0)
rows=[]
for rid,part in prepared.groupby("record_id",sort=True):
    part=part.sort_values("sample_index").reset_index(drop=True)
    for threshold in THRESHOLDS:
        pred=ivt_labels(part,threshold_dva_s=threshold)
        agreement=label_agreement(part,pred)
        rows.append({
            "record_id":rid, "participant_id":str(part.participant_id.iloc[0]).zfill(3),
            "task":part.task.iloc[0], "threshold_dva_s":threshold,
            "predicted_fixation_fraction":float(np.mean(pred=="fixation")),
            "predicted_saccade_fraction":float(np.mean(pred=="saccade")),
            "unknown_fraction":float(np.mean(pred=="unknown")),
            **agreement,
        })
sensitivity=pd.DataFrame(rows)
sensitivity.to_csv(output_dir/"06_sensitivity.csv",index=False)
display(sensitivity)


,record_id,participant_id,task,threshold_dva_s,predicted_fixation_fraction,predicted_saccade_fraction,unknown_fraction,eligible_samples,vendor_coverage_fraction,predicted_evaluable_fraction,sample_agreement
0,S_1001_S1_FXS,001,FXS,20.0,0.245871,0.014459,0.739670,3925,0.999469,0.260330,0.976306
1,S_1001_S1_FXS,001,FXS,30.0,0.924454,0.008888,0.066658,14065,0.999469,0.932878,0.995308
2,S_1001_S1_FXS,001,FXS,40.0,0.952179,0.006168,0.041653,14442,0.999469,0.957883,0.996538
3,S_1001_S1_HSS,001,HSS,20.0,0.147336,0.112252,0.740412,26296,0.999921,0.259588,0.944744
4,S_1001_S1_HSS,001,HSS,30.0,0.651191,0.103930,0.244879,76493,0.999921,0.755121,0.985018
...,...,...,...,...,...,...,...,...,...,...,...
115,S_1010_S1_RAN,010,RAN,30.0,0.731129,0.085079,0.183791,82504,0.999921,0.816209,0.977771
116,S_1010_S1_RAN,010,RAN,40.0,0.874122,0.076433,0.049445,96084,0.999921,0.950555,0.984191
117,S_1010_S1_TEX,010,TEX,20.0,0.079388,0.155658,0.764955,14173,0.999867,0.235045,0.824384
118,S_1010_S1_TEX,010,TEX,30.0,0.637705,0.133684,0.228611,46514,0.999867,0.771389,0.957325


**04. Interpret Conditional Agreement Alongside Coverage**

The recorded 40-record means are:

| I-VT threshold (dva/s) | Fixation fraction | Unknown fraction | Conditional EyeLink agreement |
| ---: | ---: | ---: | ---: |
| 20 | 0.285257 | 0.622422 | 0.909743 |
| 30 | 0.612272 | 0.305744 | 0.981307 |
| 40 | 0.786466 | 0.137801 | 0.987844 |

Each mean is the **unweighted arithmetic mean of record-level values**. The agreement mean excludes records where a comparable denominator does not exist. It is **not** pooled sample accuracy, a participant-weighted outcome or agreement calculated on an invariant set of samples.

In particular, a higher threshold can increase fixation assignments and shrink the `unknown` class, thereby changing which vendor-labelled observations enter the agreement estimate. Numerical agreement should always be accompanied by denominator and reference coverage, and should not be used to select 40 dva/s retrospectively.


In [5]:
# ============== 04. Inspect Conditional Agreement and Coverage ===============

summary=sensitivity.groupby("threshold_dva_s",as_index=False).agg(
    n_records=("record_id","size"),
    mean_predicted_fixation_fraction=("predicted_fixation_fraction","mean"),
    mean_unknown_fraction=("unknown_fraction","mean"),
    mean_conditional_vendor_agreement=("sample_agreement","mean"),
)
summary.to_csv(output_dir/"06_sensitivity_summary.csv",index=False)
display(summary)


,threshold_dva_s,n_records,mean_predicted_fixation_fraction,mean_unknown_fraction,mean_conditional_vendor_agreement
0,20.0,40,0.285257,0.622422,0.909743
1,30.0,40,0.612272,0.305744,0.981307
2,40.0,40,0.786466,0.137801,0.987844


**04A. Verify Comparison Denominators and Reconcile the 30-dva/s Baseline**

Use the existing `06_sensitivity.csv` (**120 rows**) and `03_vendor_agreement.csv` (**40 source records**) without rerunning I-VT or reloading the large prepared sample table.

For each prespecified threshold, report the total number of jointly evaluable source samples, the number of records with a defined conditional comparison, mean predicted evaluable fraction, mean vendor coverage and both the unweighted record-mean and eligible-sample-weighted conditional agreements. Both agreement measures remain **conditional on changing eligible observations**; neither is an independent accuracy estimate.

Also require the **30-dva/s record-level eligible counts, vendor coverage, predicted evaluability and conditional agreement** to match the corresponding saved values from Notebook 03. A mismatch raises an explicit error rather than silently pooling incompatible analyses.

The new `06_reference_support.csv` is a compact evidence output for Notebook 07. This check has not yet been executed on the saved original-data tables; do not report its detailed support counts as observed until it has run.


In [ ]:
# ============= 04A. Check Vendor Support and I-VT Baseline ====================

# ---------------- 1. Restore Only Compact Saved Notebook Outputs ------------
from gaze_evidence import (
    threshold_reference_support, reconcile_ivt_baseline,
)

ensure_stage_inputs(output_dir, 6, ("06_sensitivity.csv",))
ensure_stage_inputs(output_dir, 3, ("03_vendor_agreement.csv",))
saved_sensitivity = pd.read_csv(
    output_dir / "06_sensitivity.csv",
    dtype={"record_id": str, "participant_id": str},
)
original_reference = pd.read_csv(
    output_dir / "03_vendor_agreement.csv",
    dtype={"record_id": str},
)

# ---------------- 2. Reconcile I-VT at the Original 30-dva/s Setting ---------
baseline_alignment = reconcile_ivt_baseline(
    saved_sensitivity, original_reference, baseline_dva_s=30.0,
)
print("Notebook 03/06 aligned 30-dva/s source records:", len(baseline_alignment))
print("All saved conditional reference metrics agree at 30 dva/s.")

# ------------------- 3. Show Threshold-Dependent Support ----------------------
reference_support = threshold_reference_support(
    saved_sensitivity, expected_thresholds=(20.0, 30.0, 40.0),
)
reference_support.to_csv(
    output_dir / "06_reference_support.csv", index=False,
)
display(reference_support[[
    "threshold_dva_s", "n_records", "n_vendor_evaluable_records",
    "total_vendor_evaluable_samples", "mean_predicted_evaluable_fraction",
    "mean_vendor_coverage_fraction", "mean_conditional_vendor_agreement",
    "eligible_sample_weighted_agreement",
]])
print("New compact output:", output_dir / "06_reference_support.csv")


**05. Visualise the Observed Class-Assignment Sensitivity**

The saved Notebook 06 execution shows changes in average fixation and unknown sample fractions across the fixed 20/30/40-dva/s grid. The revised plot additionally shows the saccade fraction, using the already saved 120-row `06_sensitivity.csv`; **no detector rerun is necessary**.

The vertical axis is a fraction of all samples **averaged equally across records**, not a count of independent participants. The plotted method effect is descriptive: a larger fixation fraction and lower unknown fraction are not evidence that fixation detection improved. Compare the separate `06_reference_support.csv` for denominator and vendor-reference context.


In [ ]:
# ==================== 05. Visualise Threshold Sensitivity ====================

# ------------------- 1. Reuse Saved Classification Fractions -----------------
plot_support = pd.read_csv(output_dir / "06_reference_support.csv")
fig, ax = plt.subplots(figsize=(8.7, 4.5))
threshold = plot_support["threshold_dva_s"]
ax.plot(
    threshold, plot_support["mean_predicted_fixation_fraction"],
    marker="o", label="Mean fixation fraction",
)
ax.plot(
    threshold, plot_support["mean_predicted_saccade_fraction"],
    marker="^", label="Mean saccade fraction",
)
ax.plot(
    threshold, plot_support["mean_unknown_fraction"],
    marker="s", label="Mean unknown fraction",
)
ax.set(
    xlabel="Predefined I-VT threshold (dva/s)",
    ylabel="Mean sample fraction across records",
    ylim=(0, 1), title="I-VT Threshold Sensitivity (Descriptive)",
)
ax.set_xticks([20.0, 30.0, 40.0])
ax.grid(alpha=0.2)
ax.legend(loc="best")
fig.tight_layout()
save_figure(fig, output_dir / "06_threshold_sensitivity.png")
plt.show()


**06. Interpret Sensitivity and Export Current Evidence**

The recorded source analysis comprises `06_sensitivity.csv` (120 source record/threshold combinations), `06_sensitivity_summary.csv` (three record-mean summaries) and `06_threshold_sensitivity.png` (assignment fractions). The new Section 04A produces `06_reference_support.csv` for eligible sample totals, coverage and the Notebook 03 baseline reconciliation.

These outputs do **not** establish an optimal I-VT threshold, expert-validated event quality, real-time operation or a cognitive outcome. Extreme calculated speeds identified in Notebook 02 remain unresolved and may affect the interpreted event labels.

Notebook 07 will use the **compact** current evidence tables for cross-notebook consistency and synthesis. The output ZIP remains a private transfer package; do not commit generated large tables or ZIPs to GitHub.


In [ ]:
# ============== 06. Document Robustness Boundaries and Download ==============

print("Predefined thresholds:", (20.0, 30.0, 40.0))
print("Detailed source output:", output_dir / "06_sensitivity.csv")
print("Reference denominator support:", output_dir / "06_reference_support.csv")
print("Classification sensitivity plot:", output_dir / "06_threshold_sensitivity.png")
download_outputs_from_colab("eye_tracking_foundations/output", "06")
